In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from scipy.stats import poisson
import numpy as np
import requests
import os
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime
from data.fetching_data import (find_team_id, find_league_id, get_league_id, get_recent_form, find_player_id, get_fixture_stats, get_season_fixtures, 
get_matchday_fixtures, get_all_league_fixtures, get_head_to_head, get_team_squad)

load_dotenv()

API_KEY = os.getenv("SPORTS_API_KEY")
BASE_URL = "https://v3.football.api-sports.io"
Headers = {'x-apisports-key': API_KEY}
AS_OF_DATE = datetime(2024, 1, 1) 


In [3]:
def compute_team_strengths(team_fixtures, team_id, league_avg_goals_for, league_avg_goals_against):
    """
    Given a team's season fixtures, compute their attack and defense strength
    relative to the league average.
    """
    goals_for = 0
    goals_against = 0
    for f in team_fixtures:
        is_home = f['teams']['home']['id'] == team_id
        gf = f['goals']['home'] if is_home else f['goals']['away']
        ga = f['goals']['away'] if is_home else f['goals']['home']
        goals_for += gf
        goals_against += ga

    games_played = len(team_fixtures)
    avg_goals_for = goals_for / games_played
    avg_goals_against = goals_against / games_played

    attack_strength = avg_goals_for / league_avg_goals_for
    defense_strength = avg_goals_against / league_avg_goals_against

    return {"attack": attack_strength, "defense": defense_strength, "games_played": games_played}

In [4]:
def calculate_league_averages(all_league_fixtures):
    """
    Compute the league-wide average goals scored per team per game.
    Pass in every fixture in the league for the season (both teams' goals count).
    """
    total_goals = 0
    total_team_games = 0
    for f in all_league_fixtures:
        total_goals += f['goals']['home'] + f['goals']['away']
        total_team_games += 2  # each fixture = 2 team-appearances

    league_avg = total_goals / total_team_games
    return league_avg  # same baseline used for both "for" and "against"

In [5]:
def calculate_win_probability_scratch_poisson(home_strength, away_strength, league_avg_goals, max_goals=6, home_advantage=1.35):
    """
    Compute win/draw/loss probabilities using a Poisson scoreline matrix.
    home_advantage: multiplier on home team's expected goals (~1.3-1.4 is typical in real models).
    """
    home_expected_goals = league_avg_goals * home_strength["attack"] * away_strength["defense"] * home_advantage
    away_expected_goals = league_avg_goals * away_strength["attack"] * home_strength["defense"]

    # Build the scoreline probability matrix
    home_probs = [poisson.pmf(i, home_expected_goals) for i in range(max_goals + 1)]
    away_probs = [poisson.pmf(i, away_expected_goals) for i in range(max_goals + 1)]

    score_matrix = np.outer(home_probs, away_probs)

    home_win_prob = np.sum(np.tril(score_matrix, -1))  # home goals > away goals
    draw_prob = np.sum(np.diag(score_matrix))            # equal goals
    away_win_prob = np.sum(np.triu(score_matrix, 1))     # away goals > home goals

    return {
    "home_win_probability": float(round(home_win_prob * 100, 1)),
    "draw_probability": float(round(draw_prob * 100, 1)),
    "away_win_probability": float(round(away_win_prob * 100, 1)),
    "home_expected_goals": float(round(home_expected_goals, 2)),
    "away_expected_goals": float(round(away_expected_goals, 2)),
}

POISSON DECAY COLES


In [6]:
import numpy as np

def calculate_time_weight(match_timestamp, as_of_date, xi=0.0018):
    """
    xi controls how fast old matches lose influence.
    Higher xi = faster decay (recent matches dominate more).
    0.0018 is a commonly cited starting value from Dixon-Coles literature
    (roughly a half-life of a bit over a year).
    """
    days_since = (as_of_date.timestamp() - match_timestamp) / 86400
    return np.exp(-xi * days_since)

In [7]:
def compute_team_strengths_weighted(team_fixtures, team_id, league_avg_goals, as_of_date, xi=0.0018):
    weighted_goals_for = 0
    weighted_goals_against = 0
    total_weight = 0

    for f in team_fixtures:
        is_home = f['teams']['home']['id'] == team_id
        gf = f['goals']['home'] if is_home else f['goals']['away']
        ga = f['goals']['away'] if is_home else f['goals']['home']

        w = calculate_time_weight(f['fixture']['timestamp'], as_of_date, xi)
        weighted_goals_for += gf * w
        weighted_goals_against += ga * w
        total_weight += w

    avg_goals_for = weighted_goals_for / total_weight
    avg_goals_against = weighted_goals_against / total_weight

    return {
        "attack": avg_goals_for / league_avg_goals,
        "defense": avg_goals_against / league_avg_goals,
        "effective_games": total_weight,  # weighted sample size, for reference
    }

In [8]:
def dixon_coles_tau(home_goals, away_goals, home_expected, away_expected, rho=-0.13):
    """
    Adjusts probability for low-scoring results only (0-0, 1-0, 0-1, 1-1).
    rho is typically small and negative for football (~ -0.1 to -0.2).
    """
    if home_goals == 0 and away_goals == 0:
        return 1 - (home_expected * away_expected * rho)
    elif home_goals == 0 and away_goals == 1:
        return 1 + (home_expected * rho)
    elif home_goals == 1 and away_goals == 0:
        return 1 + (away_expected * rho)
    elif home_goals == 1 and away_goals == 1:
        return 1 - rho
    else:
        return 1.0  # no adjustment for other scorelines

In [9]:
from scipy.stats import poisson

def calculate_win_probability_dixon_coles(home_strength, away_strength, league_avg_goals,
                                            max_goals=6, home_advantage=1.35, rho=-0.13):
    home_expected = league_avg_goals * home_strength["attack"] * away_strength["defense"] * home_advantage
    away_expected = league_avg_goals * away_strength["attack"] * home_strength["defense"]

    score_matrix = np.zeros((max_goals + 1, max_goals + 1))
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            base_prob = poisson.pmf(h, home_expected) * poisson.pmf(a, away_expected)
            tau = dixon_coles_tau(h, a, home_expected, away_expected, rho)
            score_matrix[h][a] = base_prob * tau

    # renormalize since the tau adjustment can slightly break total probability = 1
    score_matrix = score_matrix / score_matrix.sum()

    home_win_prob = np.sum(np.tril(score_matrix, -1))
    draw_prob = np.sum(np.diag(score_matrix))
    away_win_prob = np.sum(np.triu(score_matrix, 1))

    return {
        "home_win_probability": float(round(home_win_prob * 100, 1)),
        "draw_probability": float(round(draw_prob * 100, 1)),
        "away_win_probability": float(round(away_win_prob * 100, 1)),
        "home_expected_goals": float(round(home_expected, 2)),
        "away_expected_goals": float(round(away_expected, 2)),
    }

In [5]:
def get_all_league_fixtures(season, league_id= Premier_League):
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers= Headers,
        params={"league": league_id, "season": season}
    )
    data = response.json()
    return [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

all_fixtures_2023 = get_all_league_fixtures(2023)
print(len(all_fixtures_2023))  # sanity check — should be a decent number of matches

380


In [10]:
league_avg = calculate_league_averages(all_fixtures_2023)
print(league_avg)  # typically somewhere around 1.3-1.5 for Premier League

1.6394736842105264


In [6]:
def get_season_fixtures(team_id, season= Season, league_id= Premier_League, as_of_date=AS_OF_DATE):
    """All finished fixtures for a team in a season, up to the cutoff date — no last-5 limit."""
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers= Headers,
        params={"team": team_id, "season": season, "league": league_id}
    )
    data = response.json()
    fixtures = [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

    if as_of_date is not None:
        cutoff = int(as_of_date.timestamp())
        fixtures = [f for f in fixtures if f['fixture']['timestamp'] < cutoff]

    return fixtures

In [11]:
city_fixtures = get_season_fixtures(39, season=2023)
arsenal_fixtures = get_season_fixtures(36, season=2023)

city_strength = compute_team_strengths(city_fixtures, 39, league_avg, league_avg)
arsenal_strength = compute_team_strengths(arsenal_fixtures, 36, league_avg, league_avg)

print(city_strength)
print(arsenal_strength)

{'attack': 0.9149277688603531, 'defense': 0.9454253611556982, 'games_played': 20}
{'attack': 0.8539325842696628, 'defense': 1.0674157303370786, 'games_played': 20}


In [7]:
def find_team_id(team_name, season):
    data = get_teams(season)
    matches = [
        t for t in data['response']
        if team_name.lower() in t['team']['name'].lower()
    ]
    if not matches:
        return {"error": f"No team found matching '{team_name}'"}
    if len(matches) > 1:
        return {"possible_matches": [
            {"id": t['team']['id'], "name": t['team']['name']} for t in matches
        ]}
    return {"id": matches[0]['team']['id'], "name": matches[0]['team']['name']}

def get_teams(season):
    response = requests.get(f"{BASE_URL}/teams", headers=Headers, params={"league": Premier_League, "season": season})
    return response.json()

In [8]:
find_team_id("Wolves", 2023)

{'id': 39, 'name': 'Wolves'}

In [9]:
find_team_id("Fulham", 2023)

{'id': 36, 'name': 'Fulham'}

In [14]:
result = calculate_win_probability_scratch_poisson(city_strength, arsenal_strength, league_avg)
print(result)

{'home_win_probability': 55.9, 'draw_probability': 20.5, 'away_win_probability': 22.9, 'home_expected_goals': 2.16, 'away_expected_goals': 1.32}


In [15]:
def build_fixtures_cache(team_ids, season=Season):
    cache = {}
    for team_id in team_ids:
        cache[team_id] = get_season_fixtures(team_id, season=season)
    return cache

In [16]:
teams_data = get_teams(Season)
all_team_ids = [t['team']['id'] for t in teams_data['response']]
print(len(all_team_ids))  # should be 20 for Premier League

fixtures_cache = build_fixtures_cache(all_team_ids)

20


In [23]:
def get_all_league_fixtures(season, league_id=Premier_League):
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers=Headers,
        params={"league": league_id, "season": season}
    )
    data = response.json()
    return [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

all_fixtures_2023 = get_all_league_fixtures(2023)
print(len(all_fixtures_2023))

380


In [24]:
def get_matchday_fixtures(round_name, season=Season, league_id=Premier_League):
    """
    Get all fixtures for a specific matchday/round, e.g. 'Regular Season - 21'.
    Returns finished fixtures only, with real scores included for comparison.
    """
    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers=Headers,
        params={"league": league_id, "season": season, "round": round_name}
    )
    data = response.json()
    return [f for f in data['response'] if f['fixture']['status']['short'] == 'FT']

In [6]:
cutoff = int(AS_OF_DATE.timestamp())
upcoming = [f for f in all_fixtures_2023 if f['fixture']['timestamp'] >= cutoff]
upcoming.sort(key=lambda f: f['fixture']['timestamp'])

# peek at the round name of the very next matches after the cutoff
print(upcoming[0]['league']['round'])
print(upcoming[0]['fixture']['date'])

Regular Season - 20
2024-01-01T20:00:00+00:00


In [7]:
next_matchday = get_matchday_fixtures("Regular Season - 21")  # use whatever round name you found
print(len(next_matchday))
for f in next_matchday:
    print(f['teams']['home']['name'], f['goals']['home'], '-', f['goals']['away'], f['teams']['away']['name'])

10
Burnley 1 - 1 Luton
Chelsea 1 - 0 Fulham
Newcastle 2 - 3 Manchester City
Everton 0 - 0 Aston Villa
Manchester United 2 - 2 Tottenham
Arsenal 5 - 0 Crystal Palace
Brentford 3 - 2 Nottingham Forest
Sheffield Utd 2 - 2 West Ham
Bournemouth 0 - 4 Liverpool
Brighton 0 - 0 Wolves


In [10]:
team_names = ["Burnley", "Luton", "Chelsea", "Fulham", "Newcastle", "Manchester City", "Everton", "Aston Villa", "Manchester United", 
              "Tottenham", "Arsenal", "Crystal Palace", "Brentford", "Nottingham Forest", "Sheffield Utd","West Ham", "Bournemouth", "Liverpool", "Brighton", "Wolves"]
team_ids = {}
for name in team_names:
    result = find_team_id(name, 2023)
    if "id" in result:
        team_ids[name] = result["id"]
    else:
        print(f"PROBLEM with '{name}': {result}")

print(team_ids)
    

PROBLEM with 'Liverpool': {'error': "No team found matching 'Liverpool'"}
PROBLEM with 'Brighton': {'error': "No team found matching 'Brighton'"}
PROBLEM with 'Wolves': {'error': "No team found matching 'Wolves'"}
{'Burnley': 44, 'Luton': 1359, 'Chelsea': 49, 'Fulham': 36, 'Newcastle': 34, 'Manchester City': 50, 'Everton': 45, 'Aston Villa': 66, 'Manchester United': 33, 'Tottenham': 47, 'Arsenal': 42, 'Crystal Palace': 52, 'Brentford': 55, 'Nottingham Forest': 65, 'Sheffield Utd': 62, 'West Ham': 48, 'Bournemouth': 35}


In [40]:
find_team_id("Wolves",2023)

{'id': 39, 'name': 'Wolves'}

In [46]:
raw = get_teams(2023)
print(raw['errors'])
print(raw['results'])

[]
20


In [16]:
import time

team_ids = {}
for name in team_names:
    raw = get_teams(2023)  # call it directly here to see the real error
    print(name, raw['errors'], raw['results'])
    time.sleep(15)  # slow down to avoid hitting a burst limit

Burnley [] 20
Luton [] 20
Chelsea [] 20
Fulham [] 20
Newcastle [] 20
Manchester City [] 20
Everton [] 20
Aston Villa [] 20
Manchester United [] 20
Tottenham [] 20
Arsenal [] 20
Crystal Palace [] 20
Brentford [] 20
Nottingham Forest [] 20
Sheffield Utd [] 20
West Ham [] 20
Bournemouth [] 20
Liverpool [] 20
Brighton [] 20
Wolves [] 20


In [19]:
all_teams_2023 = get_teams(2023)['response']

def find_team_id_local(team_name, all_teams):
    matches = [t for t in all_teams if team_name.lower() in t['team']['name'].lower()]
    if not matches:
        return {"error": f"No team found matching '{team_name}'"}
    if len(matches) > 1:
        return {"possible_matches": [{"id": t['team']['id'], "name": t['team']['name']} for t in matches]}
    return {"id": matches[0]['team']['id'], "name": matches[0]['team']['name']}

team_ids = {}
for name in team_names:
    result = find_team_id_local(name, all_teams_2023)
    if "id" in result:
        team_ids[name] = result["id"]
    else:
        print(f"PROBLEM with '{name}': {result}")

print(team_ids)

{'Burnley': 44, 'Luton': 1359, 'Chelsea': 49, 'Fulham': 36, 'Newcastle': 34, 'Manchester City': 50, 'Everton': 45, 'Aston Villa': 66, 'Manchester United': 33, 'Tottenham': 47, 'Arsenal': 42, 'Crystal Palace': 52, 'Brentford': 55, 'Nottingham Forest': 65, 'Sheffield Utd': 62, 'West Ham': 48, 'Bournemouth': 35, 'Liverpool': 40, 'Brighton': 51, 'Wolves': 39}


In [18]:
print(team_ids)
print(len(team_ids))

{}
0


In [20]:
import time

fixtures_cache = {}
for name, tid in team_ids.items():
    fixtures = get_season_fixtures(tid, season=2023)
    print(name, tid, len(fixtures))
    fixtures_cache[tid] = fixtures
    time.sleep(10)

Burnley 44 20
Luton 1359 19
Chelsea 49 20
Fulham 36 20
Newcastle 34 19
Manchester City 50 19
Everton 45 20
Aston Villa 66 20
Manchester United 33 20
Tottenham 47 20
Arsenal 42 20
Crystal Palace 52 20
Brentford 55 19
Nottingham Forest 65 20
Sheffield Utd 62 20
West Ham 48 19
Bournemouth 35 19
Liverpool 40 19
Brighton 51 19
Wolves 39 20


In [25]:
all_fixtures_2023 = get_all_league_fixtures(2023)
print(len(all_fixtures_2023))  # should print 380

league_avg = calculate_league_averages(all_fixtures_2023)
print(league_avg)  # should print ~1.64, like before

380
1.6394736842105264


In [26]:
matchday_results = [
    ("Burnley", "Luton", 1, 1),
    ("Chelsea", "Fulham", 1, 0),
    ("Newcastle", "Manchester City", 2, 3),
    ("Everton", "Aston Villa", 0, 0),
    ("Manchester United", "Tottenham", 2, 2),
    ("Arsenal", "Crystal Palace", 5, 0),
    ("Brentford", "Nottingham Forest", 3, 2),
    ("Sheffield Utd", "West Ham", 2, 2),
    ("Bournemouth", "Liverpool", 0, 4),
    ("Brighton", "Wolves", 0, 0),
]

def get_actual_outcome(home_goals, away_goals):
    if home_goals > away_goals:
        return "home"
    elif home_goals < away_goals:
        return "away"
    return "draw"

for home_name, away_name, hg, ag in matchday_results:
    home_id = team_ids[home_name]
    away_id = team_ids[away_name]

    home_fixtures = fixtures_cache[home_id]
    away_fixtures = fixtures_cache[away_id]

    # Plain Poisson
    home_strength_plain = compute_team_strengths(home_fixtures, home_id, league_avg, league_avg)
    away_strength_plain = compute_team_strengths(away_fixtures, away_id, league_avg, league_avg)
    plain_pred = calculate_win_probability_scratch_poisson(home_strength_plain, away_strength_plain, league_avg)

    # Dixon-Coles (time-weighted)
    home_strength_dc = compute_team_strengths_weighted(home_fixtures, home_id, league_avg, AS_OF_DATE)
    away_strength_dc = compute_team_strengths_weighted(away_fixtures, away_id, league_avg, AS_OF_DATE)
    dc_pred = calculate_win_probability_dixon_coles(home_strength_dc, away_strength_dc, league_avg)

    actual = get_actual_outcome(hg, ag)

    print(f"\n{home_name} {hg}-{ag} {away_name}  →  actual: {actual}")
    print(f"  Plain Poisson:  home={plain_pred['home_win_probability']}% draw={plain_pred['draw_probability']}% away={plain_pred['away_win_probability']}%")
    print(f"  Dixon-Coles:    home={dc_pred['home_win_probability']}% draw={dc_pred['draw_probability']}% away={dc_pred['away_win_probability']}%")


Burnley 1-1 Luton  →  actual: draw
  Plain Poisson:  home=39.9% draw=23.8% away=36.1%
  Dixon-Coles:    home=38.8% draw=26.4% away=34.7%

Chelsea 1-0 Fulham  →  actual: home
  Plain Poisson:  home=60.9% draw=18.5% away=19.3%
  Dixon-Coles:    home=60.2% draw=20.5% away=19.2%

Newcastle 2-3 Manchester City  →  actual: away
  Plain Poisson:  home=36.4% draw=21.7% away=41.4%
  Dixon-Coles:    home=35.1% draw=23.9% away=40.9%

Everton 0-0 Aston Villa  →  actual: draw
  Plain Poisson:  home=27.9% draw=22.8% away=49.0%
  Dixon-Coles:    home=27.1% draw=25.7% away=47.2%

Manchester United 2-2 Tottenham  →  actual: draw
  Plain Poisson:  home=29.2% draw=23.6% away=47.0%
  Dixon-Coles:    home=28.5% draw=26.4% away=45.1%

Arsenal 5-0 Crystal Palace  →  actual: home
  Plain Poisson:  home=71.8% draw=17.6% away=9.8%
  Dixon-Coles:    home=70.8% draw=20.0% away=9.2%

Brentford 3-2 Nottingham Forest  →  actual: home
  Plain Poisson:  home=55.0% draw=21.8% away=22.8%
  Dixon-Coles:    home=53.1% dr

In [63]:
for name, tid in team_ids.items():
    print(name, tid, len(fixtures_cache[tid]))

Burnley 44 20
Luton 1359 19
Chelsea 49 20
Fulham 36 20
Newcastle 34 19
Manchester City 50 19
Everton 45 20
Aston Villa 66 20
Manchester United 33 20
Tottenham 47 20
Arsenal 42 0
Crystal Palace 52 0
Brentford 55 0
Nottingham Forest 65 0
Sheffield Utd 62 0
West Ham 48 0
Bournemouth 35 0
Liverpool 40 0
Brighton 51 0
Wolves 39 0


**Chelsea and liverpool season 2022**

In [11]:
result_liv = find_team_id("liverpool")
print(result_liv)
result_chel = find_team_id("chelsea")
print(result_chel)

{'id': 40, 'name': 'Liverpool'}
{'id': 49, 'name': 'Chelsea'}


In [12]:
result = find_league_id("premier league")
result

{'id': 39, 'name': 'Premier League'}

In [13]:
liv = get_league_id("liverpool", 2022)
print(liv)

{'possible_leagues': [{'id': 2, 'league_name': 'UEFA Champions League'}, {'id': 39, 'league_name': 'Premier League'}]}


In [15]:
chel = get_league_id("chelsea", 2022)
print(chel)

{'possible_leagues': [{'id': 2, 'league_name': 'UEFA Champions League'}, {'id': 39, 'league_name': 'Premier League'}]}


In [16]:
liv["possible_leagues"][0]["league_name"]

'UEFA Champions League'

In [17]:
man_u = get_league_id("manchester united", 2022)
man_u

{'id': 39, 'league_name': 'Premier League'}

In [18]:
man_u["league_name"]

'Premier League'

In [20]:
liv = get_season_fixtures("liverpool", 2022, AS_OF_DATE)
liv

KeyError: 'id'

## **Testing different league matchups**

In [3]:
result_inter = find_team_id("inter")
print(result_inter)
result_athletico = find_team_id("atletico madrid")
print(result_athletico)

{'possible_matches': [{'id': 505, 'name': 'Inter'}, {'id': 3342, 'name': "Inter Club d'Escaldes"}]}
{'id': 530, 'name': 'Atletico Madrid'}


In [3]:
inter = get_season_fixtures("inter", 2022, AS_OF_DATE)
inter

{'possible_matches': [{'id': 505, 'name': 'Inter'},
  {'id': 3342, 'name': "Inter Club d'Escaldes"}]}

In [9]:
ATM = get_season_fixtures("atletico madrid", 2022, AS_OF_DATE)
ATM

[{'teams': {'home': {'id': 546, 'name': 'Getafe'},
   'away': {'id': 530, 'name': 'Atletico Madrid'}},
  'goals': {'home': 0, 'away': 3},
  'timestamp': 1660584600},
 {'teams': {'home': {'id': 530, 'name': 'Atletico Madrid'},
   'away': {'id': 533, 'name': 'Villarreal'}},
  'goals': {'home': 0, 'away': 2},
  'timestamp': 1661103000},
 {'teams': {'home': {'id': 532, 'name': 'Valencia'},
   'away': {'id': 530, 'name': 'Atletico Madrid'}},
  'goals': {'home': 0, 'away': 1},
  'timestamp': 1661803200},
 {'teams': {'home': {'id': 548, 'name': 'Real Sociedad'},
   'away': {'id': 530, 'name': 'Atletico Madrid'}},
  'goals': {'home': 1, 'away': 1},
  'timestamp': 1662222600},
 {'teams': {'home': {'id': 530, 'name': 'Atletico Madrid'},
   'away': {'id': 538, 'name': 'Celta Vigo'}},
  'goals': {'home': 4, 'away': 1},
  'timestamp': 1662836400},
 {'teams': {'home': {'id': 530, 'name': 'Atletico Madrid'},
   'away': {'id': 541, 'name': 'Real Madrid'}},
  'goals': {'home': 1, 'away': 2},
  'timesta

In [5]:
import inspect
print(inspect.getsource(get_season_fixtures))

def get_season_fixtures(team_name: str, season: int, as_of_date=AS_OF_DATE) -> dict:
    """All finished fixtures for a team in a season, up to the cutoff date."""

    league = resolve_league_for_team(team_name, season)
    if "id" not in league:
        return league  # error or unresolved ambiguity — bail out cleanly
    league_id = league["id"]

    team_result = find_team_id(team_name)
    if "error" in team_result:
        return {"error": "Could not find team"}
    if "possible_matches" in team_result:
        return team_result  # let the agent see the ambiguity and ask the user to clarify
    team_id = team_result["id"]

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT home_team_id, home_team_name, away_team_id, away_team_name,home_goals, away_goals, timestamp FROM fixtures WHERE (home_team_id = ? OR away_team_id = ?) AND (season = ?) AND (league_id = ?) AND timestamp < ?",
                   (team_id, team_id, season, league_id, int(as

In [4]:
inter = get_matchday_fixtures ("inter", 2022, 34)

In [5]:
inter

{'possible_matches': [{'id': 505, 'name': 'Inter'},
  {'id': 3342, 'name': "Inter Club d'Escaldes"}]}